# 01 - Data cleaning



**Purpose:** load the 15 raw CSV files, fix data-quality problems, check that the tables join correctly and save clean copies for the analysis and dashboard notebooks.

**Input:** `Forkly/Data/*.csv` (raw files, never edited)
**Output:** `Forkly/Data_clean/*.csv` and `Forkly/Data_clean/cleaning_log.csv`



## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# works whether the notebook runs from the Forkly folder or from Forkly/notebooks
BASE = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent
RAW_DIR = BASE / "Data"
CLEAN_DIR = BASE / "Data_clean"
CLEAN_DIR.mkdir(exist_ok=True)

# every fix gets one line here, so the changes are easy to report later
log = []

def log_fix(table, issue, rows_affected, action):
    log.append({"table": table, "issue": issue, "rows_affected": int(rows_affected), "action": action})
    print(f"[{table}] {issue}: {rows_affected:,} rows -> {action}")

## 2. Load raw data

In [2]:
DATE_COLS = {
    "calendar": ["date"],
    "cities": ["launch_date"],
    "weather_daily": ["date"],
    "restaurants": ["onboarded_date", "offboarded_date"],
    "couriers": ["join_date"],
    "promotions": ["start_date", "end_date"],
    "subscriptions": ["start_date", "end_date"],
    "orders": ["order_datetime"],
    "deliveries": ["pickup_datetime", "dropoff_datetime"],
    "reviews": ["review_datetime"],
    "support_tickets": ["created_datetime", "resolved_datetime"],
    "marketing_spend": ["date"],
}
# customers.signup_date stays as text for now, it has two date formats

raw = {}
for path in sorted(RAW_DIR.glob("*.csv")):
    raw[path.stem] = pd.read_csv(path, parse_dates=DATE_COLS.get(path.stem, []))

summary = pd.DataFrame({
    "rows": {k: len(v) for k, v in raw.items()},
    "columns": {k: v.shape[1] for k, v in raw.items()},
    "missing_cells": {k: int(v.isna().sum().sum()) for k, v in raw.items()},
    "duplicate_rows": {k: int(v.duplicated().sum()) for k, v in raw.items()},
}).sort_values("rows", ascending=False)
summary

,rows,columns,missing_cells,duplicate_rows
order_items,662291,6,2648,0
orders,301200,19,801183,1146
deliveries,282862,13,2012,0
reviews,132720,8,47765,0
customers,45000,8,3117,0
weather_daily,21912,5,0,0
menu_items,18454,5,0,0
subscriptions,15287,8,17832,0
support_tickets,13365,10,0,0
marketing_spend,10956,6,25,0


In [3]:
# work on copies so the raw data stays untouched in memory
df = {k: v.copy() for k, v in raw.items()}

## 3. Orders
Checks: inconsistent text in `platform` and `payment_method`, duplicated rows, and that `total_amount` adds up.

In [4]:
orders = df["orders"]
print(orders["platform"].value_counts(dropna=False), "\n")
print(orders["payment_method"].value_counts(dropna=False))

platform
Android    132066
iOS        131832
Web         33729
ios          1590
android      1568
web           415
Name: count, dtype: int64 

payment_method
Card              145385
Apple Pay          64176
Google Pay         42737
Cash               28089
Forkly Wallet      18396
CARD                1208
APPLE PAY            511
GOOGLE PAY           349
CASH                 209
FORKLY WALLET        140
Name: count, dtype: int64


In [5]:
platform_map = {"ios": "iOS", "android": "Android", "web": "Web"}
before = orders["platform"].copy()
orders["platform"] = orders["platform"].str.strip().str.lower().map(platform_map)
log_fix("orders", "platform in wrong case", (before != orders["platform"]).sum(), "mapped to iOS / Android / Web")

payment_map = {"card": "Card", "apple pay": "Apple Pay", "google pay": "Google Pay",
               "cash": "Cash", "forkly wallet": "Forkly Wallet"}
before = orders["payment_method"].copy()
orders["payment_method"] = orders["payment_method"].str.strip().str.lower().map(payment_map)
log_fix("orders", "payment_method uppercase / trailing space", (before != orders["payment_method"]).sum(),
        "trimmed and mapped to standard names")

assert orders["platform"].notna().all() and orders["payment_method"].notna().all()
orders["payment_method"].value_counts()

[orders] platform in wrong case: 3,573 rows -> mapped to iOS / Android / Web
[orders] payment_method uppercase / trailing space: 2,417 rows -> trimmed and mapped to standard names


payment_method
Card             146593
Apple Pay         64687
Google Pay        43086
Cash              28298
Forkly Wallet     18536
Name: count, dtype: int64

In [6]:
# text is fixed first: some duplicated rows differ only by casing, so they match after this step
n_dup = orders.duplicated().sum()
orders = orders.drop_duplicates().reset_index(drop=True)
log_fix("orders", "duplicated rows", n_dup, "dropped")

# order_id should now be unique
assert orders["order_id"].is_unique
print(f"Orders after removing duplicates: {len(orders):,}")

[orders] duplicated rows: 1,200 rows -> dropped
Orders after removing duplicates: 300,000


In [7]:
# total = subtotal + delivery fee + service fee - discount + tip
calc_total = (orders["items_subtotal"] + orders["delivery_fee"] + orders["service_fee"]
              - orders["discount_amount"] + orders["tip_amount"]).round(2)
mismatch = (calc_total - orders["total_amount"]).abs() > 0.02
print(f"Orders where total_amount does not add up: {mismatch.sum()}")
print(orders.loc[mismatch, "order_status"].value_counts())

# all of them are cancelled orders: the tip was removed but the old total (with tip) was kept
orders.loc[mismatch, "total_amount"] = calc_total[mismatch]
log_fix("orders", "total_amount still includes tip on cancelled orders", mismatch.sum(),
        "recalculated from its components")

# types and helper columns used later
orders["promo_id"] = orders["promo_id"].astype("Int64")
orders["has_promo"] = orders["promo_id"].notna()
orders["is_cancelled"] = orders["order_status"].eq("Cancelled")
orders["order_date"] = orders["order_datetime"].dt.normalize()
orders["year"] = orders["order_datetime"].dt.year
orders["month"] = orders["order_datetime"].dt.to_period("M").dt.to_timestamp()
orders["hour"] = orders["order_datetime"].dt.hour

print(orders["order_status"].value_counts())
print(orders.groupby("year").size())
df["orders"] = orders

Orders where total_amount does not add up: 4853
order_status
Cancelled    4853
Name: count, dtype: int64
[orders] total_amount still includes tip on cancelled orders: 4,853 rows -> recalculated from its components
order_status
Delivered          281698
Cancelled           17138
Failed delivery      1164
Name: count, dtype: int64
year
2021    36000
2022    60000
2023    61000
2024    69000
2025    74000
dtype: int64


## 4. Customers
Checks: mixed date formats in `signup_date`, impossible ages, inconsistent `gender` values.

In [8]:
customers = df["customers"]

iso = pd.to_datetime(customers["signup_date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(customers["signup_date"], format="%d/%m/%Y", errors="coerce")
log_fix("customers", "signup_date in DD/MM/YYYY format", (iso.isna() & dmy.notna()).sum(), "parsed separately")
customers["signup_date"] = iso.fillna(dmy)
assert customers["signup_date"].notna().all()
customers["signup_date"].describe()

[customers] signup_date in DD/MM/YYYY format: 884 rows -> parsed separately


count                         45000
mean     2023-01-30 12:54:43.200000
min             2021-01-01 00:00:00
25%             2021-12-04 00:00:00
50%             2022-10-31 00:00:00
75%             2024-04-02 00:00:00
max             2025-12-31 00:00:00
Name: signup_date, dtype: object

In [9]:
print(customers["age"].describe())

bad_age = customers["age"].notna() & ~customers["age"].between(16, 100)
print("\nImpossible ages:", sorted(customers.loc[bad_age, "age"].unique()))
log_fix("customers", "age outside 16-100", bad_age.sum(), "set to missing")
log_fix("customers", "age missing in raw file", customers["age"].isna().sum(), "kept missing, age_group = Unknown")
customers.loc[bad_age, "age"] = np.nan
customers["age"] = customers["age"].astype("Int64")

customers["age_group"] = pd.cut(customers["age"].astype("float"), bins=[15, 24, 34, 44, 54, 64, 100],
                                labels=["16-24", "25-34", "35-44", "45-54", "55-64", "65+"])
customers["age_group"] = customers["age_group"].cat.add_categories("Unknown").fillna("Unknown")
customers["age_group"].value_counts().sort_index()

count    42315.000000
mean        46.351719
std         16.892511
min          0.000000
25%         38.000000
50%         45.000000
75%         53.000000
max        999.000000
Name: age, dtype: float64

Impossible ages: [np.float64(0.0), np.float64(7.0), np.float64(121.0), np.float64(150.0), np.float64(999.0)]
[customers] age outside 16-100: 45 rows -> set to missing
[customers] age missing in raw file: 2,685 rows -> kept missing, age_group = Unknown


age_group
16-24        272
25-34       6090
35-44      14334
45-54      12198
55-64       6139
65+         3237
Unknown     2730
Name: count, dtype: int64

In [10]:
print(customers["gender"].value_counts(dropna=False))

gender_map = {"female": "Female", "f": "Female", "male": "Male", "m": "Male", "other": "Other", "o": "Other"}
before = customers["gender"].copy()
customers["gender"] = customers["gender"].str.strip().str.lower().map(gender_map).fillna("Unknown")
log_fix("customers", "gender spelled F / female / blank", (before != customers["gender"]).sum(),
        "mapped to Female / Male / Other / Unknown")
customers["gender"].value_counts()

gender
Female    21998
Male      19501
Other       859
F           675
M           631
female      448
NaN         432
male        420
O            22
other        14
Name: count, dtype: int64
[customers] gender spelled F / female / blank: 2,642 rows -> mapped to Female / Male / Other / Unknown


gender
Female     23121
Male       20552
Other        895
Unknown      432
Name: count, dtype: int64

In [11]:
# a customer can't order before signing up
first_order = df["orders"].groupby("customer_id")["order_date"].min()
signup = customers["customer_id"].map(first_order)
early = signup.notna() & (signup < customers["signup_date"])
print(customers.loc[early, ["customer_id", "signup_date"]].assign(first_order=signup[early]))

customers.loc[early, "signup_date"] = signup[early]
log_fix("customers", "first order before signup_date", early.sum(), "signup_date moved to first order date")

customers["signup_year"] = customers["signup_date"].dt.year
customers["signup_month"] = customers["signup_date"].dt.to_period("M").dt.to_timestamp()
df["customers"] = customers

       customer_id signup_date first_order
17136       117137  2022-05-08  2022-05-01
32349       132350  2024-02-02  2024-02-01
36673       136674  2024-09-13  2024-09-01
[customers] first order before signup_date: 3 rows -> signup_date moved to first order date


## 5. Restaurants
Checks: `cuisine` has uppercase variants and trailing spaces.

In [12]:
restaurants = df["restaurants"]
print("Distinct cuisine values before:", restaurants["cuisine"].nunique())

# use the most common spelling of each cuisine as the standard one
clean_key = restaurants["cuisine"].str.strip().str.lower()
standard = (restaurants.assign(key=clean_key)
            .groupby("key")["cuisine"].agg(lambda s: s.str.strip().value_counts().idxmax()))
before = restaurants["cuisine"].copy()
restaurants["cuisine"] = clean_key.map(standard)
log_fix("restaurants", "cuisine casing / trailing spaces", (before != restaurants["cuisine"]).sum(),
        "standardised to the most common spelling")

print("Distinct cuisine values after:", restaurants["cuisine"].nunique())
restaurants["cuisine"].value_counts()

Distinct cuisine values before: 47
[restaurants] cuisine casing / trailing spaces: 238 rows -> standardised to the most common spelling
Distinct cuisine values after: 14


cuisine
Pizza                 455
Burgers               408
Asian                 323
Sushi                 306
Kebab                 239
Lithuanian/Local      196
Italian               188
Healthy/Salads        172
Indian                121
Mexican               112
Desserts              112
Grocery               112
Breakfast & Coffee     96
Georgian               49
Name: count, dtype: int64

In [13]:
# offboarded restaurants should not be marked active
print(pd.crosstab(restaurants["is_active"], restaurants["offboarded_date"].notna(),
                  rownames=["is_active"], colnames=["has_offboarded_date"]))
df["restaurants"] = restaurants

has_offboarded_date  False  True 
is_active                        
False                    0     45
True                  2844      0


## 6. Order items
Checks: missing `unit_price` and `line_total`.

In [14]:
order_items = df["order_items"].merge(df["orders"][["order_id", "year"]], on="order_id", how="left")
missing_price = order_items["unit_price"].isna()
print(f"Missing unit_price: {missing_price.sum():,} of {len(order_items):,}")

# prices are fixed per item within a year, so fill with that item's price in the same year
item_year_price = order_items.groupby(["item_id", "year"])["unit_price"].transform("median")
order_items["unit_price"] = order_items["unit_price"].fillna(item_year_price)

# fall back to menu price x yearly inflation if the item was never priced that year
if order_items["unit_price"].isna().any():
    menu = df["menu_items"].set_index("item_id")["base_price_2021"]
    infl = (order_items.dropna(subset=["unit_price"])
            .assign(ratio=lambda d: d["unit_price"] / d["item_id"].map(menu))
            .groupby("year")["ratio"].median())
    gap = order_items["unit_price"].isna()
    order_items.loc[gap, "unit_price"] = (order_items.loc[gap, "item_id"].map(menu)
                                          * order_items.loc[gap, "year"].map(infl)).round(2)

order_items["line_total"] = (order_items["quantity"] * order_items["unit_price"]).round(2)
log_fix("order_items", "missing unit_price / line_total", missing_price.sum(),
        "filled with same item's price that year, line_total recalculated")
assert order_items["unit_price"].notna().all()

Missing unit_price: 1,324 of 662,291
[order_items] missing unit_price / line_total: 1,324 rows -> filled with same item's price that year, line_total recalculated


In [15]:
# line totals should add up to the order subtotal
check = (order_items.groupby("order_id")["line_total"].sum()
         .to_frame("items_sum").join(df["orders"].set_index("order_id")["items_subtotal"]))
check["diff"] = (check["items_sum"] - check["items_subtotal"]).abs()
print(f"Orders where items do not add up to subtotal (> 2 cents): {(check['diff'] > 0.02).sum()}")
df["order_items"] = order_items.drop(columns="year")

Orders where items do not add up to subtotal (> 2 cents): 0


## 7. Deliveries
Checks: impossible `delivery_minutes`, missing `courier_id`.

In [16]:
deliveries = df["deliveries"].merge(df["orders"][["order_id", "order_datetime"]], on="order_id", how="left")
deliveries["delivery_minutes"].describe(percentiles=[0.01, 0.5, 0.99])

count    282862.000000
mean         34.406987
std          22.865245
min         -12.000000
1%           15.700000
50%          32.900000
99%          62.900000
max        1440.000000
Name: delivery_minutes, dtype: float64

In [17]:
# recalculate from timestamps: dropoff - order time
from_timestamps = ((deliveries["dropoff_datetime"] - deliveries["order_datetime"])
                   .dt.total_seconds() / 60).round(1)

bad_minutes = (deliveries["delivery_minutes"] <= 0) | (deliveries["delivery_minutes"] > 180)
print("Bad values:", deliveries.loc[bad_minutes, "delivery_minutes"].value_counts().to_dict())

deliveries.loc[bad_minutes, "delivery_minutes"] = from_timestamps[bad_minutes]
log_fix("deliveries", "delivery_minutes <= 0 or > 180", bad_minutes.sum(),
        "recalculated from order and dropoff timestamps")

# failed deliveries never reached the customer, so no delivery time
failed = deliveries["delivery_status"].eq("Failed")
deliveries["is_late"] = deliveries["is_late"].astype("boolean")  # nullable, allows missing
deliveries.loc[failed, "delivery_minutes"] = np.nan
deliveries.loc[failed, "is_late"] = pd.NA
deliveries["delivery_minutes"].describe()

Bad values: {-12.0: 50, 0.0: 47, 999.0: 46, 480.0: 42, 1440.0: 35}
[deliveries] delivery_minutes <= 0 or > 180: 220 rows -> recalculated from order and dropoff timestamps


count    281698.000000
mean         34.023190
std           9.844786
min           7.200000
25%          27.100000
50%          32.900000
75%          39.700000
max         118.000000
Name: delivery_minutes, dtype: float64

In [18]:
missing_courier = deliveries["courier_id"].isna()
log_fix("deliveries", "missing courier_id", missing_courier.sum(), "kept, flagged with courier_known = False")
deliveries["courier_id"] = deliveries["courier_id"].astype("Int64")
deliveries["courier_known"] = ~missing_courier

deliveries["minutes_vs_promise"] = (deliveries["delivery_minutes"] - deliveries["promised_minutes"]).round(1)
df["deliveries"] = deliveries.drop(columns="order_datetime")

[deliveries] missing courier_id: 848 rows -> kept, flagged with courier_known = False


## 8. Reviews
Checks: ratings outside the 1-5 scale.

In [19]:
reviews = df["reviews"]
for col in ["food_rating", "delivery_rating"]:
    bad = ~reviews[col].between(1, 5)
    log_fix("reviews", f"{col} outside 1-5", bad.sum(), "set to missing")
    reviews.loc[bad, col] = np.nan
    reviews[col] = reviews[col].astype("Int64")

reviews[["food_rating", "delivery_rating"]].describe()

[reviews] food_rating outside 1-5: 90 rows -> set to missing
[reviews] delivery_rating outside 1-5: 0 rows -> set to missing


,food_rating,delivery_rating
count,132630.0,132720.0
mean,4.222499,4.242111
std,0.758687,0.791007
min,1.0,1.0
25%,4.0,4.0
50%,4.0,4.0
75%,5.0,5.0
max,5.0,5.0


In [20]:
df["reviews"] = reviews

## 9. Support tickets
Checks: tickets resolved before they were created.

In [21]:
tickets = df["support_tickets"]
wrong_order = tickets["resolved_datetime"] < tickets["created_datetime"]
tickets.loc[wrong_order, "resolved_datetime"] = (tickets.loc[wrong_order, "created_datetime"]
                                                 + pd.to_timedelta(tickets.loc[wrong_order, "resolution_hours"], unit="h")).dt.floor("s")
log_fix("support_tickets", "resolved_datetime before created_datetime", wrong_order.sum(),
        "recalculated as created + resolution_hours")
assert (tickets["resolved_datetime"] >= tickets["created_datetime"]).all()
df["support_tickets"] = tickets

[support_tickets] resolved_datetime before created_datetime: 53 rows -> recalculated as created + resolution_hours


## 10. Marketing spend
Checks: missing `spend_eur` values.

In [22]:
mkt = df["marketing_spend"].sort_values(["channel", "date"]).reset_index(drop=True)
missing_spend = mkt["spend_eur"].isna()
print(mkt.loc[missing_spend, ["date", "channel"]].head())

# spend moves smoothly day to day, so fill gaps by interpolating within each channel
mkt["spend_eur"] = (mkt.groupby("channel")["spend_eur"]
                    .transform(lambda s: s.interpolate(limit_direction="both")).round(2))
log_fix("marketing_spend", "missing spend_eur", missing_spend.sum(), "interpolated within channel")
df["marketing_spend"] = mkt

           date        channel
212  2021-08-01  App Store Ads
1149 2024-02-24  App Store Ads
1165 2024-03-11  App Store Ads
1794 2025-11-30  App Store Ads
1938 2021-04-23     Influencer
[marketing_spend] missing spend_eur: 25 rows -> interpolated within channel


## 11. Other tables
Calendar, cities, weather, menu items, couriers, promotions and subscriptions had no planted issues. Quick sanity checks only.

In [23]:
for name in ["calendar", "cities", "weather_daily", "menu_items", "couriers", "promotions", "subscriptions"]:
    t = df[name]
    print(f"{name:15s} rows={len(t):>7,}  duplicates={t.duplicated().sum()}  missing={t.isna().sum().sum()}")

# subscriptions: blank end_date means still active
print(df["subscriptions"].groupby("status")["end_date"].apply(lambda s: s.isna().mean()))

calendar        rows=  1,826  duplicates=0  missing=0
cities          rows=     12  duplicates=0  missing=0
weather_daily   rows= 21,912  duplicates=0  missing=0
menu_items      rows= 18,454  duplicates=0  missing=0
couriers        rows=  5,369  duplicates=0  missing=0
promotions      rows=    120  duplicates=0  missing=0
subscriptions   rows= 15,287  duplicates=0  missing=17832
status
Active       1.0
Cancelled    0.0
Name: end_date, dtype: float64


## 12. Referential integrity

In [24]:
def orphans(child, child_col, parent, parent_col):
    keys = df[child][child_col].dropna()
    return int((~keys.isin(df[parent][parent_col])).sum())

checks = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "restaurant_id", "restaurants", "restaurant_id"),
    ("orders", "city_id", "cities", "city_id"),
    ("orders", "promo_id", "promotions", "promo_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "item_id", "menu_items", "item_id"),
    ("deliveries", "order_id", "orders", "order_id"),
    ("deliveries", "courier_id", "couriers", "courier_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("support_tickets", "order_id", "orders", "order_id"),
    ("subscriptions", "customer_id", "customers", "customer_id"),
    ("restaurants", "city_id", "cities", "city_id"),
    ("couriers", "city_id", "cities", "city_id"),
    ("menu_items", "restaurant_id", "restaurants", "restaurant_id"),
]
integrity = pd.DataFrame([(f"{c}.{cc}", f"{p}.{pc}", orphans(c, cc, p, pc)) for c, cc, p, pc in checks],
                         columns=["foreign_key", "references", "orphan_rows"])
assert integrity["orphan_rows"].sum() == 0, "some keys point to missing rows"
integrity

,foreign_key,references,orphan_rows
0,orders.customer_id,customers.customer_id,0
1,orders.restaurant_id,restaurants.restaurant_id,0
2,orders.city_id,cities.city_id,0
3,orders.promo_id,promotions.promo_id,0
4,order_items.order_id,orders.order_id,0
5,order_items.item_id,menu_items.item_id,0
6,deliveries.order_id,orders.order_id,0
7,deliveries.courier_id,couriers.courier_id,0
8,reviews.order_id,orders.order_id,0
9,support_tickets.order_id,orders.order_id,0


In [25]:
# business rules
o = df["orders"]
d = df["deliveries"]
print("Cancelled orders with a delivery:", d["order_id"].isin(o.loc[o["is_cancelled"], "order_id"]).sum())
print("Cancelled orders with a review:  ", df["reviews"]["order_id"].isin(o.loc[o["is_cancelled"], "order_id"]).sum())
print("Negative money values in orders: ",
      (o[["items_subtotal", "delivery_fee", "service_fee", "discount_amount", "tip_amount", "total_amount"]] < 0).sum().sum())

Cancelled orders with a delivery: 0
Cancelled orders with a review:   0
Negative money values in orders:  0


## 13. Save clean data

In [26]:
for name, table in df.items():
    table.to_csv(CLEAN_DIR / f"{name}.csv", index=False)

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(CLEAN_DIR / "cleaning_log.csv", index=False)

after = pd.DataFrame({"raw_rows": {k: len(v) for k, v in raw.items()},
                      "clean_rows": {k: len(v) for k, v in df.items()},
                      "missing_cells_raw": {k: int(v.isna().sum().sum()) for k, v in raw.items()},
                      "missing_cells_clean": {k: int(v.isna().sum().sum()) for k, v in df.items()}})
print(f"Saved {len(df)} tables to {CLEAN_DIR.resolve()}")
after.sort_values("raw_rows", ascending=False)

Saved 15 tables to C:\Users\algir\Desktop\Projects\Python\Forkly\Data_clean


,raw_rows,clean_rows,missing_cells_raw,missing_cells_clean
order_items,662291,662291,2648,0
orders,301200,300000,801183,797994
deliveries,282862,282862,2012,5504
reviews,132720,132720,47765,47855
customers,45000,45000,3117,2730
weather_daily,21912,21912,0,0
menu_items,18454,18454,0,0
subscriptions,15287,15287,17832,17832
support_tickets,13365,13365,0,0
marketing_spend,10956,10956,25,0


In [27]:
cleaning_log

,table,issue,rows_affected,action
0,orders,platform in wrong case,3573,mapped to iOS / Android / Web
1,orders,payment_method uppercase / trailing space,2417,trimmed and mapped to standard names
2,orders,duplicated rows,1200,dropped
3,orders,total_amount still includes tip on cancelled o...,4853,recalculated from its components
4,customers,signup_date in DD/MM/YYYY format,884,parsed separately
5,customers,age outside 16-100,45,set to missing
6,customers,age missing in raw file,2685,"kept missing, age_group = Unknown"
7,customers,gender spelled F / female / blank,2642,mapped to Female / Male / Other / Unknown
8,customers,first order before signup_date,3,signup_date moved to first order date
9,restaurants,cuisine casing / trailing spaces,238,standardised to the most common spelling


### Notes
- Missing values left on purpose: `age` (shown as *Unknown* age group), `courier_id` (flagged), ratings that were out of range, `promo_id` (order had no promo), cancellation fields on delivered orders, `end_date` on active subscriptions, delivery time on failed deliveries.
- Added helper columns: `orders.year / month / order_date / hour / has_promo / is_cancelled`, `customers.age_group / signup_year / signup_month`, `deliveries.courier_known / minutes_vs_promise`.
- Read the clean files with the same `parse_dates` list as section 2 (plus `customers.signup_date`).